# 2D DnCNN — Ultra-Low-Dose CT Denoising
**A complete, modular training pipeline optimised for RTX 4090.**

| Component | Detail |
|---|---|
| **Dataset** | Memory-mapped `.npy` loader — indexes every 2D slice across all patients |
| **Model** | 17-layer DnCNN with residual-learning denoiser |
| **Precision** | BFloat16 mixed-precision AMP + TF32 (no GradScaler needed) |
| **Noise** | Synthetic Gaussian noise injected on-the-fly |
| **Checkpointing** | Every 10 epochs; includes optimizer state & metrics |

---


## 1. Imports & Global Setup

In [25]:
import os
import glob
import math
from pathlib import Path
from typing import Optional, Union

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib

# Use non-interactive backend so figures are saved as PNGs (not pop-ups).
# Change to 'TkAgg' if you want interactive windows.
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Enable TF32 on Ampere+ GPUs (RTX 4090) ──────────────────────────────
# TF32 gives near-FP32 accuracy with tensor-core throughput for matmuls
# and cuDNN convolutions.
if torch.cuda.is_available():
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.8 GB


## 2. Configuration

In [26]:
# ── Paths ────────────────────────────────────────────────────────────────
DATA_DIR       = "./Model_Ready_Data/COVID-S1"   # folder of .npy patient volumes
CHECKPOINT_DIR = "./checkpoints"
VISUALS_DIR    = "./training_visuals"

# ── DataLoader ───────────────────────────────────────────────────────────
BATCH_SIZE  = 16
NUM_WORKERS = 0 if os.name == "nt" else 8
PIN_MEMORY  = torch.cuda.is_available()

# ── Training ─────────────────────────────────────────────────────────────
NUM_EPOCHS    = 100
LEARNING_RATE = 1e-5
NOISE_FACTOR  = 0.15   # Gaussian noise σ (data assumed normalised to [0, 1])

# ── Device ───────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("⚠️  CUDA not available — training will be very slow on CPU.")
else:
    print(f"🚀 Using: {torch.cuda.get_device_name(0)}")

print(f"DataLoader workers: {NUM_WORKERS}")
print(f"Pin memory: {PIN_MEMORY}")


🚀 Using: NVIDIA GeForce RTX 4090
DataLoader workers: 0
Pin memory: True


## 3. Dataset — `CT2DDataset`

Scans a directory of 3-D `.npy` volumes (shape `(D, 512, 512)`) and builds a
flat slice-level index so each `__getitem__` returns a single `[1, 512, 512]`
float32 tensor.  Volumes are opened with `mmap_mode='r'` — only the requested
page is loaded into RAM, keeping memory usage near zero even for large datasets.


In [27]:
class CT2DDataset(Dataset):
    """
    Memory-efficient 2D slice dataset built from a folder of 3D .npy volumes.

    Each .npy file is expected to have shape (D, 512, 512) where D is the
    number of axial slices for that patient.  The dataset indexes every valid
    2D slice across *all* patients so that each __getitem__ call returns a
    single [1, 512, 512] float32 tensor.

    Parameters
    ----------
    data_dir : str or Path
        Directory containing .npy patient volume files.
    """

    def __init__(self, data_dir: str):
        super().__init__()
        self.data_dir = Path(data_dir)

        self.npy_files = sorted(glob.glob(str(self.data_dir / "*.npy")))
        if len(self.npy_files) == 0:
            raise FileNotFoundError(
                f"No .npy files found in '{self.data_dir}'. "
                "Check your DATA_DIR path."
            )

        # Build a flat index → (file_idx, slice_idx) mapping.
        # Opening in mmap_mode='r' to read shape uses almost zero RAM.
        self.slice_map: list[tuple[int, int]] = []
        self.shapes:    list[tuple[int, int, int]] = []

        print(f"[CT2DDataset] Scanning {len(self.npy_files)} volumes …")
        for file_idx, fpath in enumerate(self.npy_files):
            vol = np.load(fpath, mmap_mode="r")
            num_slices = vol.shape[0]
            self.shapes.append(vol.shape)
            for s in range(num_slices):
                self.slice_map.append((file_idx, s))

        print(
            f"[CT2DDataset] Ready — {len(self.npy_files)} patients, "
            f"{len(self.slice_map):,} total 2D slices."
        )

    def __len__(self) -> int:
        return len(self.slice_map)

    def __getitem__(self, idx: int) -> torch.Tensor:
        """Returns a single slice as Tensor [1, 512, 512], dtype float32."""
        file_idx, slice_idx = self.slice_map[idx]
        vol      = np.load(self.npy_files[file_idx], mmap_mode="r")
        slice_2d = np.array(vol[slice_idx], dtype=np.float32)   # (512, 512)
        return torch.from_numpy(slice_2d).unsqueeze(0)           # [1, 512, 512]


## 4. Model — `DnCNN2D`

17-layer residual-learning denoiser.

| Layer | Operation |
|---|---|
| 1 | Conv(1→64, 3×3) → ReLU *(no BN)* |
| 2 – 16 | Conv(64→64, 3×3) → BatchNorm → ReLU |
| 17 | Conv(64→1, 3×3) *(no BN, no ReLU)* |

**Forward pass:** `output = input − network(input)` (noise residual subtraction)


In [28]:
class DnCNN2D(nn.Module):
    """
    17-layer DnCNN for 2D image denoising via residual learning.

    Forward pass (residual learning):
        noise_mask = network(x)
        output     = x - noise_mask     ← clean estimate
    """

    def __init__(self, depth: int = 17, num_features: int = 64):
        super().__init__()
        assert depth >= 3, "DnCNN needs at least 3 layers."

        layers: list[nn.Module] = []

        # Layer 1: Conv → ReLU (no BN)
        layers.append(nn.Conv2d(1, num_features, kernel_size=3, padding=1, bias=False))
        layers.append(nn.ReLU(inplace=True))

        # Layers 2 … depth-1: Conv → BN → ReLU
        for _ in range(depth - 2):
            layers.append(nn.Conv2d(num_features, num_features, kernel_size=3, padding=1, bias=False))
            layers.append(nn.BatchNorm2d(num_features))
            layers.append(nn.ReLU(inplace=True))

        # Layer 17: Conv only (no BN, no ReLU)
        layers.append(nn.Conv2d(num_features, 1, kernel_size=3, padding=1, bias=False))

        self.dncnn = nn.Sequential(*layers)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor [B, 1, H, W]  — noisy input

        Returns
        -------
        clean : Tensor [B, 1, H, W]  — denoised output (input − noise)
        """
        noise_mask = self.dncnn(x)
        return x - noise_mask


# ── Quick sanity check ────────────────────────────────────────────────────
model = DnCNN2D(depth=17, num_features=64).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"🧠 DnCNN2D — {total_params:,} parameters")

dummy = torch.randn(2, 1, 512, 512, device=device)
with torch.no_grad():
    out = model(dummy)
print(f"   Input shape : {tuple(dummy.shape)}")
print(f"   Output shape: {tuple(out.shape)}  ✅")
del dummy, out


🧠 DnCNN2D — 556,032 parameters
   Input shape : (2, 1, 512, 512)
   Output shape: (2, 1, 512, 512)  ✅


## 5. Utility Functions

In [29]:
def add_synthetic_radiation_noise(
    clean_tensor: torch.Tensor,
    noise_factor: float = 0.15,
) -> torch.Tensor:
    """
    Add random Gaussian noise to simulate ultra-low-dose CT degradation.

    Parameters
    ----------
    clean_tensor : Tensor [B, 1, H, W]  — values assumed in [0, 1]
    noise_factor : float                — std-dev of the additive Gaussian noise

    Returns
    -------
    noisy : Tensor [B, 1, H, W], clamped to [0, 1]
    """
    noise = torch.randn_like(clean_tensor) * noise_factor
    return torch.clamp(clean_tensor + noise, 0.0, 1.0)


def calculate_psnr(mse_loss: float) -> float:
    """
    Convert MSE → PSNR (dB).  Assumes max pixel value = 1.0.

    Returns +inf if mse_loss == 0.
    """
    if mse_loss == 0:
        return float("inf")
    return 10.0 * math.log10(1.0 / mse_loss)


def display_training_visuals(
    noisy:    torch.Tensor,
    denoised: torch.Tensor,
    clean:    torch.Tensor,
    epoch:    int,
    save_dir: str = VISUALS_DIR,
) -> None:
    """
    Save a 4-panel diagnostic figure for the first sample in the batch.

    Panels: Noisy Input | Extracted Noise Mask | Denoised Output | Clean GT
    """
    os.makedirs(save_dir, exist_ok=True)

    noisy_np    = noisy[0, 0].detach().cpu().numpy()
    denoised_np = denoised[0, 0].detach().cpu().numpy()
    clean_np    = clean[0, 0].detach().cpu().numpy()
    noise_mask  = noisy_np - denoised_np

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    panels = [
        ("Noisy Input",          noisy_np,    "gray"),
        ("Extracted Noise Mask", noise_mask,  "seismic"),
        ("Denoised Output",      denoised_np, "gray"),
        ("Clean Ground Truth",   clean_np,    "gray"),
    ]
    for ax, (title, img, cmap) in zip(axes, panels):
        ax.imshow(img, cmap=cmap,
                  vmin=0.0 if cmap == "gray" else None,
                  vmax=1.0 if cmap == "gray" else None)
        ax.set_title(title, fontsize=12)
        ax.axis("off")

    fig.suptitle(f"Epoch {epoch} — Training Visuals", fontsize=14, fontweight="bold")
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"epoch_{epoch:04d}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  📊 Visual saved → {save_path}")


## 6. Training Loop

**Key design choices:**
- **BFloat16 autocast** — no `GradScaler` needed (BF16 has the same exponent range as FP32).
- **Synthetic noise on-the-fly** — gives virtually infinite noisy/clean pairs.
- **MSELoss** between denoised output and clean ground truth.
- **Checkpoints** saved every 10 epochs (includes optimizer state + metrics).


In [30]:
def train_dncnn(
    model: nn.Module,
    dataloader: DataLoader,
    num_epochs: int = NUM_EPOCHS,
    lr: float = LEARNING_RATE,
    noise_factor: float = NOISE_FACTOR,
    checkpoint_dir: str = CHECKPOINT_DIR,
    device: Optional[Union[torch.device, str]] = None,
    resume_checkpoint: Optional[str] = None,
) -> None:
    """Train DnCNN and optionally resume from a saved checkpoint."""
    os.makedirs(checkpoint_dir, exist_ok=True)

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    if device.type == "cuda" and not torch.cuda.is_available():
        print("CUDA requested but not available; falling back to CPU.")
        device = torch.device("cpu")

    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    start_epoch = 1
    if resume_checkpoint is not None:
        if not os.path.exists(resume_checkpoint):
            raise FileNotFoundError(f"Checkpoint not found: {resume_checkpoint}")

        ckpt = torch.load(resume_checkpoint, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        loaded_epoch = int(ckpt.get("epoch", 0))
        start_epoch = loaded_epoch + 1
        print(
            f"Resumed from: {resume_checkpoint} "
            f"(loaded epoch {loaded_epoch}, next epoch {start_epoch})"
        )

        if start_epoch > num_epochs:
            print(
                f"Checkpoint epoch {loaded_epoch} is already >= target "
                f"num_epochs={num_epochs}. Nothing to train."
            )
            return

    amp_status = "CUDA bfloat16 AMP" if device.type == "cuda" else "FP32"
    print("=" * 70)
    print(f"Training DnCNN2D | target epochs={num_epochs} | LR={lr}")
    print(f"Device: {device} | {amp_status}")
    print(f"Noise sigma: {noise_factor} | Batch size: {dataloader.batch_size}")
    print(f"Starting from epoch: {start_epoch}")
    print("=" * 70)

    for epoch in range(start_epoch, num_epochs + 1):
        model.train()
        epoch_loss_sum = 0.0
        num_batches = 0
        noisy_batch = denoised_batch = clean_batch = None

        pbar = tqdm(
            dataloader,
            desc=f"Epoch {epoch}/{num_epochs}",
            unit="batch",
            leave=True,
        )

        for clean_batch in pbar:
            clean_batch = clean_batch.to(device, non_blocking=True)
            noisy_batch = add_synthetic_radiation_noise(clean_batch, noise_factor)

            optimizer.zero_grad(set_to_none=True)
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    denoised_batch = model(noisy_batch)
                    loss = criterion(denoised_batch, clean_batch)
            else:
                denoised_batch = model(noisy_batch)
                loss = criterion(denoised_batch, clean_batch)

            loss.backward()
            optimizer.step()

            batch_mse = loss.item()
            epoch_loss_sum += batch_mse
            num_batches += 1
            pbar.set_postfix({"MSE": f"{batch_mse:.6f}"})

        if num_batches == 0:
            print(
                f"Epoch {epoch}/{num_epochs} produced zero batches. "
                "Check dataset size, batch_size, and drop_last settings."
            )
            continue

        avg_mse = epoch_loss_sum / num_batches
        avg_psnr = calculate_psnr(avg_mse)
        print(
            f"\nEpoch {epoch}/{num_epochs} | "
            f"Avg MSE: {avg_mse:.6f} | Avg PSNR: {avg_psnr:.2f} dB\n"
        )

        display_training_visuals(noisy_batch, denoised_batch, clean_batch, epoch)

        if epoch % 10 == 0:
            ckpt_path = os.path.join(checkpoint_dir, f"dncnn2d_epoch_{epoch:04d}.pth")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "avg_mse": avg_mse,
                    "avg_psnr": avg_psnr,
                },
                ckpt_path,
            )
            print(f"Checkpoint saved: {ckpt_path}\n")

## 7. Build Dataset & DataLoader

In [31]:
dataset = CT2DDataset(data_dir=DATA_DIR)

dataloader_kwargs = dict(
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,   # avoid tiny last batch (BN stability)
)

if NUM_WORKERS > 0:
    dataloader_kwargs.update(
        persistent_workers=True,  # keep workers alive between epochs
        prefetch_factor=2,        # each worker prefetches 2 batches
    )

dataloader = DataLoader(dataset, **dataloader_kwargs)

# One-batch smoke test to catch worker issues early
sample_batch = next(iter(dataloader))
print(f"Sample batch shape: {tuple(sample_batch.shape)}")
del sample_batch

print(f"Batches per epoch: {len(dataloader):,}")


[CT2DDataset] Scanning 104 volumes …
[CT2DDataset] Ready — 104 patients, 14,974 total 2D slices.
Sample batch shape: (16, 1, 512, 512)
Batches per epoch: 935


## 8. Train

In [32]:
RESUME_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "dncnn2d_epoch_0080.pth")
if not os.path.exists(RESUME_CHECKPOINT):
    raise FileNotFoundError(f"Expected checkpoint not found: {RESUME_CHECKPOINT}")

train_dncnn(
    model=model,
    dataloader=dataloader,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    noise_factor=NOISE_FACTOR,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    resume_checkpoint=RESUME_CHECKPOINT,
)

print("\n🏁 Training complete.")


Resumed from: ./checkpoints\dncnn2d_epoch_0080.pth (loaded epoch 80, next epoch 81)
Training DnCNN2D | target epochs=100 | LR=1e-05
Device: cuda | CUDA bfloat16 AMP
Noise sigma: 0.15 | Batch size: 16
Starting from epoch: 81


Epoch 81/100:   0%|          | 0/935 [00:00<?, ?batch/s]


Epoch 81/100 | Avg MSE: 0.001331 | Avg PSNR: 28.76 dB

  📊 Visual saved → ./training_visuals\epoch_0081.png


Epoch 82/100:   0%|          | 0/935 [00:00<?, ?batch/s]


Epoch 82/100 | Avg MSE: 0.001340 | Avg PSNR: 28.73 dB

  📊 Visual saved → ./training_visuals\epoch_0082.png


...


Epoch 98/100 | Avg MSE: 0.001286 | Avg PSNR: 28.91 dB

  📊 Visual saved → ./training_visuals\epoch_0098.png


Epoch 99/100:   0%|          | 0/935 [00:00<?, ?batch/s]


Epoch 99/100 | Avg MSE: 0.001311 | Avg PSNR: 28.82 dB

  📊 Visual saved → ./training_visuals\epoch_0099.png


Epoch 100/100:   0%|          | 0/935 [00:00<?, ?batch/s]


Epoch 100/100 | Avg MSE: 0.001308 | Avg PSNR: 28.83 dB

  📊 Visual saved → ./training_visuals\epoch_0100.png
Checkpoint saved: ./checkpoints\dncnn2d_epoch_0100.pth


🏁 Training complete.


## 9. Inference Helper (optional)

Load a saved checkpoint and denoise a single slice.


In [33]:
def load_checkpoint(checkpoint_path: str, device: torch.device) -> DnCNN2D:
    """Load a DnCNN2D model from a .pth checkpoint."""
    ckpt  = torch.load(checkpoint_path, map_location=device)
    model = DnCNN2D(depth=17, num_features=64).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}  "
          f"(PSNR: {ckpt['avg_psnr']:.2f} dB)")
    return model


def denoise_slice(
    model:      DnCNN2D,
    noisy_np:   np.ndarray,   # shape (512, 512), values in [0, 1]
    device:     torch.device,
) -> np.ndarray:
    """Run the model on a single noisy 2D slice and return a numpy array."""
    tensor = torch.from_numpy(noisy_np).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        denoised = model(tensor)
    return denoised[0, 0].cpu().numpy()


# ── Example usage (uncomment after training) ─────────────────────────────
CKPT = "./checkpoints/dncnn2d_epoch_0200.pth"
inference_model = load_checkpoint(CKPT, device)

# Load one slice from the dataset
ds = CT2DDataset(data_dir='')
sample_clean = dataset[0].numpy()[0]               # (512, 512)
sample_noisy = sample_clean + np.random.randn(*sample_clean.shape) * NOISE_FACTOR
sample_noisy = np.clip(sample_noisy, 0.0, 1.0)

sample_denoised = denoise_slice(inference_model, sample_noisy, device)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img) in zip(axes, [
    ("Noisy Input", sample_noisy),
    ("Denoised",    sample_denoised),
    ("Clean GT",    sample_clean),
]):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


FileNotFoundError: [Errno 2] No such file or directory: './checkpoints/dncnn2d_epoch_0200.pth'